# Hint-AUC quick start (Colab)

Deterministic hints and Hint-AUC for line art colorization ([paper](https://doi.org/10.1109/TVCG.2026.3738401), [code](https://github.com/MADONOKOUKI/DetFill-with-DHT)). Runs on CPU; no GPU needed.

1. install `hintauc` and run the built-in demo
2. generate deterministic hints for an illustration (an example, or your own upload)
3. score a colorization and compute Hint-AUC

In [ ]:
!pip -q install "hintauc>=0.3.4" matplotlib

## 1. Built-in demo

A synthetic illustration, its hints at 8 ratios, an oracle colorizer and the Hint-AUC (about 10 s).

In [ ]:
!hintauc demo

## 2. Deterministic hints for an illustration

The example below is the illustration used in the README. To use your own, set `UPLOAD = True` (a colour illustration; it is segmented at the resolution you upload).

In [ ]:
import urllib.request

UPLOAD = False
RAW = "https://raw.githubusercontent.com/MADONOKOUKI/DetFill-with-DHT/main/assets/readme"
urllib.request.urlretrieve(f"{RAW}/ex_gt.png", "illustration.png")
urllib.request.urlretrieve(f"{RAW}/ex_sketch.png", "line_art.png")
urllib.request.urlretrieve(f"{RAW}/ex_detfill_r10.png", "detfill_10.png")
image_path = "illustration.png"
if UPLOAD:
    from google.colab import files
    image_path = next(iter(files.upload()))
print(image_path)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import hintauc

# size=64 is the paper's hint resolution; path_method="geodesic" is the fast, bit-reproducible path
hints = hintauc.generate_hints(image_path, size=64, path_method="geodesic")
print("regions:", hints.n_regions())

rgb = lambda bgr: cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
ratios = [0.01, 0.10, 0.50, 1.00]
fig, ax = plt.subplots(1, len(ratios) + 1, figsize=(15, 3))
ax[0].imshow(rgb(cv2.imread(image_path))); ax[0].set_title("input")
for a, r in zip(ax[1:], ratios):
    color, mask = hints.at_ratio(r, hint_type="scribble", resize_to=256)
    a.imshow(rgb(color)); a.set_title(f"{r:.0%} of regions")
for a in ax: a.axis("off")
plt.show()

The hints depend only on the illustration: running the cell again gives the same hints. `hints.save("out/illustration")` writes them in the DetFill data layout.

## 3. Scoring

Compare one colorization with its ground truth (here DetFill's output with 10 % hints, from the README):

In [ ]:
ev = hintauc.Evaluator(metrics=("mse", "psnr", "ssim"))
print(ev("detfill_10.png", "illustration.png"))

Hint-AUC of a colorizer over the whole range of hint ratios. Replace `hint_fill_colorizer` (an oracle reference that fills each hinted region with its hint colour; not a model) with a function that wraps your model: it receives `line_art`, `hint_color` (BGR) and `hint_mask` and returns a BGR image. See [docs/evaluate_your_model.md](https://github.com/MADONOKOUKI/DetFill-with-DHT/blob/main/docs/evaluate_your_model.md).

In [ ]:
res = hintauc.evaluate_colorizer(
    hintauc.hint_fill_colorizer, [("line_art.png", "illustration.png")],
    metrics=("mse", "psnr", "ssim"), path_method="geodesic", oracle=True)
print("Hint-AUC:", res["hint_auc"])

alphas = sorted(res["per_alpha"])
plt.plot(alphas, [res["per_alpha"][a]["psnr"] for a in alphas], "o-")
plt.xlabel("hint ratio"); plt.ylabel("PSNR"); plt.title("oracle hint-fill baseline")
plt.show()

Next: [DetFill inference](https://github.com/MADONOKOUKI/DetFill-with-DHT/tree/main/detfill) (GPU), [reproduce the paper](https://github.com/MADONOKOUKI/DetFill-with-DHT/tree/main/reproduce), [online demo](https://huggingface.co/spaces/madorin0130/Hint-AUC).